# 8. Pydantic & Data Validation

Pydantic is one of the most important libraries in modern Python backend and AI work.

It helps you:
- define data models clearly
- validate incoming data automatically
- convert JSON into Python objects
- enforce constraints before business logic runs
- generate JSON Schema for APIs and tools

In backend and AI systems, validation is critical because bad input is one of the biggest sources of bugs.

This notebook follows the same pattern as the rest of the course:
1. explain the concept
2. show the practical idea
3. give runnable examples


## 1. Models

A model is the core idea in Pydantic. It is a Python class that defines the shape of data.

You declare fields with types, and Pydantic validates the values automatically.

This is the foundation of strongly typed data handling in APIs and AI workflows.


In [1]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    age: int

user = User(id=1, name="Alice", age=30)
print(user)
print(user.model_dump())

# Theory:
# - BaseModel turns a Python class into a validated schema
# - field types are enforced automatically
# - invalid values raise validation errors before business logic runs


id=1 name='Alice' age=30
{'id': 1, 'name': 'Alice', 'age': 30}


## 2. Field Validation

Each field can have strict rules. Pydantic validates field values based on their type and constraints.

This is important because APIs often receive data from outside systems, and validation is the first defensive layer.


In [2]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(min_length=2, max_length=20)
    price: float = Field(gt=0)
    stock: int = Field(ge=0)

product = Product(name="Laptop", price=999.99, stock=5)
print(product)

try:
    Product(name="A", price=-1, stock=-2)
except Exception as e:
    print(type(e).__name__)
    print(e)

# Theory:
# - Field() adds constraints like min_length, gt, ge
# - invalid inputs are rejected early
# - this keeps domain rules clear and consistent


name='Laptop' price=999.99 stock=5
ValidationError
3 validation errors for Product
name
  String should have at least 2 characters [type=string_too_short, input_value='A', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short
price
  Input should be greater than 0 [type=greater_than, input_value=-1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
stock
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-2, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal


## 3. Nested Models

Real APIs often send nested JSON objects. Pydantic makes nested data validation straightforward.

A model can contain other models, and Pydantic will validate each level automatically.


In [3]:
from pydantic import BaseModel, Field

class Address(BaseModel):
    city: str
    zip_code: str

class Customer(BaseModel):
    name: str
    address: Address
    tags: list[str] = []

customer = Customer(
    name="Bob",
    address={"city": "Paris", "zip_code": "75000"},
    tags=["vip", "beta"]
)

print(customer)
print(customer.model_dump())

# Theory:
# - nested models validate child objects as part of the parent model
# - dictionaries can be automatically converted into model instances
# - this mirrors JSON payloads in real APIs


name='Bob' address=Address(city='Paris', zip_code='75000') tags=['vip', 'beta']
{'name': 'Bob', 'address': {'city': 'Paris', 'zip_code': '75000'}, 'tags': ['vip', 'beta']}


## 4. Custom Validators

Sometimes the built-in types are not enough. Pydantic lets you define custom validation logic for a field or model.

This is useful when business rules are specific, such as email format, password rules, or ID shape.


In [4]:
from pydantic import BaseModel, field_validator

class UserProfile(BaseModel):
    username: str
    email: str

    @field_validator('username')
    @classmethod
    def username_must_be_valid(cls, value: str) -> str:
        if ' ' in value:
            raise ValueError('username cannot contain spaces')
        return value

    @field_validator('email')
    @classmethod
    def email_must_contain_at_symbol(cls, value: str) -> str:
        if '@' not in value:
            raise ValueError('email must contain @')
        return value

profile = UserProfile(username='alice', email='alice@example.com')
print(profile)

try:
    UserProfile(username='alice smith', email='bad-email')
except Exception as e:
    print(type(e).__name__)
    print(e)

# Theory:
# - field_validator lets you add domain-specific logic
# - custom rules run after type coercion and before the object is accepted
# - this is ideal for complex input rules


username='alice' email='alice@example.com'
ValidationError
2 validation errors for UserProfile
username
  Value error, username cannot contain spaces [type=value_error, input_value='alice smith', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
email
  Value error, email must contain @ [type=value_error, input_value='bad-email', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


## 5. Serialization and Deserialization

Serialization means converting a model to a Python dictionary or JSON string.

Deserialization means creating a model from raw data such as a dictionary or JSON payload.

This is the core of API request/response work.


In [5]:
from pydantic import BaseModel
import json

class Event(BaseModel):
    id: int
    name: str
    active: bool = True

payload = {"id": 7, "name": "launch", "active": True}

event = Event(**payload)
print('Deserialized:', event)
print('Dictionary:', event.model_dump())
print('JSON:', event.model_dump_json())

# Example of parsing from JSON string
json_text = '{"id": 9, "name": "deploy", "active": false}'
parsed = Event.model_validate_json(json_text)
print('Parsed from JSON:', parsed)

# Theory:
# - BaseModel accepts Python dicts and JSON-like data
# - model_dump() turns a model into a dictionary
# - model_dump_json() turns it into a JSON string
# - APIs use this heavily for request/response handling


Deserialized: id=7 name='launch' active=True
Dictionary: {'id': 7, 'name': 'launch', 'active': True}
JSON: {"id":7,"name":"launch","active":true}
Parsed from JSON: id=9 name='deploy' active=False


## 6. JSON Schema

Pydantic can generate JSON Schema from your models.

This is incredibly useful because the same schema can be reused by frontend code, API documentation, and AI tools that need structured input definitions.


In [6]:
from pydantic import BaseModel

class Article(BaseModel):
    title: str
    author: str
    published: bool = False
    words: int = 0

print(Article.model_json_schema())

# Theory:
# - JSON Schema describes the expected data format in a standard way
# - it can be used by APIs, validators, docs, and client SDKs
# - Pydantic generates this directly from your model


{'properties': {'title': {'title': 'Title', 'type': 'string'}, 'author': {'title': 'Author', 'type': 'string'}, 'published': {'default': False, 'title': 'Published', 'type': 'boolean'}, 'words': {'default': 0, 'title': 'Words', 'type': 'integer'}}, 'required': ['title', 'author'], 'title': 'Article', 'type': 'object'}


## 7. Optional and Default Fields

Not every field is required. Some fields are optional, while others have default values.

This makes models flexible but still controlled.


In [7]:
from typing import Optional
from pydantic import BaseModel

class Profile(BaseModel):
    username: str
    bio: Optional[str] = None
    age: int = 18
    is_active: bool = True

print(Profile(username='sam'))
print(Profile(username='sam', bio='Python learner'))

# Theory:
# - Optional[str] means the field may be missing or set to None
# - default values allow partial payloads
# - required fields remain explicit while optional ones are more flexible


username='sam' bio=None age=18 is_active=True
username='sam' bio='Python learner' age=18 is_active=True


## 8. Strict Validation

Strict validation means the value must match the expected type exactly, without coercion when possible.

This is important when you care about correctness and do not want Python to silently convert values in surprising ways.


In [9]:
from typing_extensions import Annotated
from pydantic import BaseModel, ConfigDict, Strict

class StrictUser(BaseModel):
    model_config = ConfigDict(strict=True)
    age: int
    name: str

# This example shows strict mode behavior; coercion is not allowed
try:
    StrictUser(age='30', name='Alice')
except Exception as e:
    print(type(e).__name__)
    print(e)

# Also possible at field level using Annotated + Strict
class AnotherUser(BaseModel):
    age: Annotated[int, Strict()]
    name: Annotated[str, Strict()]

print(AnotherUser(age=30, name='Alice'))

# Theory:
# - strict mode rejects unsafe type coercion
# - this is useful for correctness-sensitive systems
# - you often combine strictness with domain validation


ValidationError
1 validation error for StrictUser
age
  Input should be a valid integer [type=int_type, input_value='30', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_type
age=30 name='Alice'


## 9. Configuration

Pydantic models can be configured globally. Configuration controls things like strictness, aliasing, and validation behavior.

This helps you define expectations at the model level rather than repeating them everywhere.


In [10]:
from pydantic import BaseModel, ConfigDict, Field

class Settings(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True, validate_assignment=True)
    username: str = Field(min_length=3)
    email: str

settings = Settings(username='  alice  ', email='alice@example.com')
print(settings)

settings.username = 'bob'
print(settings)

# Theory:
# - ConfigDict lets you control validation behavior globally
# - str_strip_whitespace removes unwanted spaces automatically
# - validate_assignment re-validates when a field is reassigned


username='alice' email='alice@example.com'
username='bob' email='alice@example.com'


## 10. Type-Driven Validation

Pydantic is strongest when data types communicate intent. The model itself defines the contract.

This makes the code more readable, easier to maintain, and safer when used in APIs or AI prompt systems.


In [12]:
%pip install email-validator

from pydantic import BaseModel, EmailStr

class Contact(BaseModel):
    name: str
    email: EmailStr
    score: float

contact = Contact(name='Dana', email='dana@example.com', score=4.8)
print(contact)

try:
    Contact(name='Eve', email='not-an-email', score=1.2)
except Exception as e:
    print(type(e).__name__)
    print(e)

# Theory:
# - your type annotations become validation rules
# - this reduces boilerplate and makes data contracts explicit
# - Pydantic helps turn Python types into trusted runtime validation


Note: you may need to restart the kernel to use updated packages.
name='Dana' email='dana@example.com' score=4.8
ValidationError
1 validation error for Contact
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='not-an-email', input_type=str]



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
